# Analysis & reporting of preprocessed recordings

Select a single culture or a group of cultures (from the same experiment or across experiments) and
generate an MXtreme PDF report.

The report is built to answer two questions in order:

1. **how is the group developing?** — one page per topic, showing the across-culture mean ± SEM over a
   faint line per culture, so an outlier or a dying culture never hides inside the average;
2. **what is each culture doing?** — two pages per culture: the recording itself over DIV (ASDR and
   burst origins), then the distributions behind the summary statistics (CDFs, one line per DIV).

This notebook uses the `WaveTraining` experiment, which is a *phased* experiment: each recording is
split into `pre` / `train` / `post` windows. A report always plots exactly one phase, so this is also
where phase selection is demonstrated.

Assumes the store already holds preprocessed `.npz` files and burst CSVs (see
`preprocess_pipeline.ipynb` and `burst_detection.ipynb`).

In [2]:
# Imports
from mxtreme import io
from mxtreme.config import Config
from mxtreme.identity import CultureID, CultureSelector
from mxtreme.analysis import generate_report

### 1. Configure the output directory

MXtreme output is saved to `root_path`, which can be fed directly when creating a `Config` object or
read from `mxtreme.toml`.

Here we read cleaned npzs from `preprocessed/` and burst data from `burst_data/`, and save output to
`analysis/` — all under the store's `root_path`.

In [3]:
config = Config.from_toml("mxtreme.toml")
print("managed store:", config.data_root.resolve())
print("analysis dir: ", config.analysis_dir.resolve())

managed store: /Users/cgrass04/Code/mxtreme-dev/mxtreme-claude/claude_output
analysis dir:  /Users/cgrass04/Code/mxtreme-dev/mxtreme-claude/claude_output/analysis


Selections are resolved against the store's registry (`registry.csv`), which preprocessing keeps
current as it writes. If you have copied recordings into the store by hand — as with the example data
here — rebuild it from what is actually on disk:

In [4]:
io.rebuild_registry(config)

Registry rebuilt from 75 recordings -> ../../mxtreme-claude/claude_output/registry.csv


75

### 2. Pick the cultures

`WaveTraining` ran on two MaxTwo chips. Five of its cultures have both a preprocessed `.npz` and a
burst CSV, over a ragged set of DIVs (`M07462` well0 stops at DIV36; the rest run to DIV43):

In [5]:
CULTURES = [
    CultureID("WaveTraining", "M07459", "0"),
    CultureID("WaveTraining", "M07459", "1"),
    CultureID("WaveTraining", "M07459", "2"),
    CultureID("WaveTraining", "M07462", "0"),
    CultureID("WaveTraining", "M07462", "2"),
]

group = CultureSelector(cultures=CULTURES)

### 3. Define the report sections

`sections` selects what goes in the PDF:

- `"activity"`    — spiking activity: firing rate / ISI / spike amplitude / active electrodes.
- `"bursting"`    — burst statistics: IBI, size, duration, rate.
- `"cultures"`    — the two per-culture pages, for every culture in the selection.
- `"stimulation"` — stimulation delivered per DIV (skips cultures that were never stimulated).
- `"performance"` — a learning curve from a user-defined objective (default: burst direction).

The default is `("activity", "bursting", "cultures")`.

In [6]:
SECTIONS = ("activity", "bursting", "cultures")

### 4. Group report

A `CultureSelector` gives population-level (mean ± SEM across cultures) panels with each culture drawn
faintly underneath, followed by the per-culture pages.

The first run computes and caches a per-culture summary CSV and a distribution `.npz` for each
culture; re-running is much faster.

In [7]:
generate_report(group, config, sections=SECTIONS)

Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well0/WaveTraining_M07459_well0_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well1/WaveTraining_M07459_well1_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well2/WaveTraining_M07459_well2_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07462/well0/WaveTraining_M07462_well0_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07462/well2/WaveTraining_M07462_well2_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well0/WaveTraining_M07459_well0_burst_activity_summary.csv
Loading existing s

PosixPath('../../mxtreme-claude/claude_output/analysis/reports/group_5cultures_report.pdf')

### 5. Single culture

A `CultureID` expands to all DIVs on record for that culture. The same sections apply — the group
pages simply have one culture in them.

In [ ]:
generate_report(CultureID("WaveTraining", "M07459", "2"), config, sections=SECTIONS)

### 6. Choosing a phase

Exactly one phase is plotted per report. With no `phase` argument the first phase in the sequence is
used — `full` for recordings with no user-supplied phases, and `pre` for this experiment. Pass a name
to report on a different window; asking for a phase the data doesn't have fails immediately rather
than producing empty panels.

In [8]:
generate_report(
    group, config,
    sections=SECTIONS,
    phase="pre",
    output_path=config.analysis_dir / "reports" / "WaveTraining_group_train.pdf",
)

Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well0/WaveTraining_M07459_well0_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well1/WaveTraining_M07459_well1_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well2/WaveTraining_M07459_well2_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07462/well0/WaveTraining_M07462_well0_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07462/well2/WaveTraining_M07462_well2_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well0/WaveTraining_M07459_well0_burst_activity_summary.csv
Loading existing s

PosixPath('../../mxtreme-claude/claude_output/analysis/reports/WaveTraining_group_train.pdf')

### 7. Measuring performance against a user-defined objective

The performance section's objective is user-defined: pass `objective_fn(burst_df) -> float`. The
default scores burst propagation toward the side each culture was trained to burst from. Supply your
own for a task-specific readout — e.g. mean burst size as a crude excitability proxy:

In [9]:
def mean_size_objective(burst_df):
    return float(burst_df["size_frac_elec"].mean()) if len(burst_df) else float("nan")


generate_report(
    group, config,
    sections=("performance",),
    objective_fn=mean_size_objective,
    output_path=config.analysis_dir / "reports" / "WaveTraining_custom_objective.pdf",
)

Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well0/WaveTraining_M07459_well0_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well1/WaveTraining_M07459_well1_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07459/well2/WaveTraining_M07459_well2_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07462/well0/WaveTraining_M07462_well0_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/activity/WaveTraining/M07462/well2/WaveTraining_M07462_well2_channel_activity_summary.csv
Loading existing summary from ../../mxtreme-claude/claude_output/analysis/performance/WaveTraining/M07459/well0/WaveTraining_M07459_well0_performance_summary.csv
Loading existing s

PosixPath('../../mxtreme-claude/claude_output/analysis/reports/WaveTraining_custom_objective.pdf')